# Step 5C.3 — GeNIS temporal boundary audit (діагностика, БЕЗ фіксації спліту)
Мета: знайти РЕАЛЬНІ часові межі, де і train, і val, і test містять ОБИДВА класи
(Normal і Attack), замість наївного 70/15/15 за кількістю рядків (який дав val/test
з 0% Normal). Не будує моделі, не фіксує остаточний train_end/val_end.

Ядро: IDS Research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
genis = pd.read_parquet(ENRICHED_DIR / "genis_with_cve_context.parquet",
                        columns=["StartTime", "SubCategoryLabel"])

is_attack = ~genis["SubCategoryLabel"].astype(str).str.lower().str.startswith("benign")
genis["_is_attack"] = is_attack.astype(int)
t = pd.to_datetime(genis["StartTime"])
genis["_date"] = t.dt.date
genis["_hour"] = t.dt.floor("h")

print(f"GeNIS: {len(genis):,} flows, діапазон {t.min()} — {t.max()}")
print(f"Normal={int((~is_attack).sum()):,}  Attack={int(is_attack.sum()):,}")

GeNIS: 1,472,133 flows, діапазон 2025-02-06 09:00:44 — 2025-02-12 16:42:04
Normal=35,006  Attack=1,437,127


In [2]:
# ---------- 1. Розподіл по ДНЯХ ----------
daily = genis.groupby("_date").agg(
    n_total=("_is_attack", "size"),
    n_attack=("_is_attack", "sum"),
).assign(n_normal=lambda d: d["n_total"] - d["n_attack"],
         attack_rate=lambda d: d["n_attack"] / d["n_total"])
print("Розподіл по днях (n_total / n_normal / n_attack / attack_rate):")
display(daily[["n_total", "n_normal", "n_attack", "attack_rate"]])

Розподіл по днях (n_total / n_normal / n_attack / attack_rate):


,n_total,n_normal,n_attack,attack_rate
_date,,,,
2025-02-06,4096,4096,0,0.0
2025-02-07,18048,18048,0,0.0
2025-02-08,12862,12862,0,0.0
2025-02-10,22022,0,22022,1.0
2025-02-11,1347284,0,1347284,1.0
2025-02-12,67821,0,67821,1.0


In [3]:
# ---------- 2. Розподіл по ГОДИНАХ — де саме Normal з'являється/зникає ----------
hourly = genis.groupby("_hour").agg(
    n_total=("_is_attack", "size"),
    n_attack=("_is_attack", "sum"),
).assign(n_normal=lambda d: d["n_total"] - d["n_attack"],
         attack_rate=lambda d: d["n_attack"] / d["n_total"])

hours_with_normal = hourly[hourly["n_normal"] > 0]
print(f"Годин з ХОЧА Б ОДНИМ Normal flow: {len(hours_with_normal)} з {len(hourly)}")
print(f"\nПерша година з Normal: {hours_with_normal.index.min()}")
print(f"Остання година з Normal: {hours_with_normal.index.max()}")
print(f"\nОстання година в датасеті загалом: {hourly.index.max()}")
print(f"Розрив між останнім Normal і кінцем датасету: "
      f"{hourly.index.max() - hours_with_normal.index.max()}")

Годин з ХОЧА Б ОДНИМ Normal flow: 25 з 38

Перша година з Normal: 2025-02-06 09:00:00
Остання година з Normal: 2025-02-08 16:00:00

Остання година в датасеті загалом: 2025-02-12 16:00:00
Розрив між останнім Normal і кінцем датасету: 4 days 00:00:00


In [4]:
# ---------- 3. Кандидатні межі: train_end / val_end так, щоб val і test МАЛИ Normal ----------
# Обираємо межі не за квантилем кількості рядків, а за ЧАСОМ, що ЗАЛИШАЄ Normal-приклади
# у val і test. Пробуємо кілька варіантів принципу "останні N% ЧАСОВОГО ДІАПАЗОНУ (не рядків)".
t_min, t_max = t.min(), t.max()
total_span = t_max - t_min

for train_frac, val_frac in [(0.70, 0.15), (0.60, 0.20), (0.50, 0.25)]:
    train_end = t_min + total_span * train_frac
    val_end = t_min + total_span * (train_frac + val_frac)

    train_mask = t < train_end
    val_mask = (t >= train_end) & (t < val_end)
    test_mask = t >= val_end

    def summarize(mask, label):
        n = int(mask.sum())
        a = int(is_attack[mask].sum())
        return f"{label}: n={n:,}  normal={n-a:,}  attack={a:,}  attack_rate={a/n:.4%}" if n else f"{label}: n=0 (ПОРОЖНЬО)"

    print(f"\n--- Часові частки (не квантилі рядків): train_end={train_end}, val_end={val_end} ---")
    print(" ", summarize(train_mask, "train"))
    print(" ", summarize(val_mask, "val"))
    print(" ", summarize(test_mask, "test"))
    both_classes_ok = all(
        int(is_attack[m].sum()) > 0 and int((~is_attack[m]).sum()) > 0
        for m in [train_mask, val_mask, test_mask] if m.sum() > 0
    )
    print(f"  Обидва класи присутні у ВСІХ непорожніх сегментах: {both_classes_ok}")


--- Часові частки (не квантилі рядків): train_end=2025-02-10 19:11:40, val_end=2025-02-11 17:56:52 ---
  train: n=57,028  normal=35,006  attack=22,022  attack_rate=38.6161%
  val: n=1,347,284  normal=0  attack=1,347,284  attack_rate=100.0000%
  test: n=67,821  normal=0  attack=67,821  attack_rate=100.0000%
  Обидва класи присутні у ВСІХ непорожніх сегментах: False

--- Часові частки (не квантилі рядків): train_end=2025-02-10 04:01:32, val_end=2025-02-11 10:21:48 ---
  train: n=35,006  normal=35,006  attack=0  attack_rate=0.0000%
  val: n=74,607  normal=0  attack=74,607  attack_rate=100.0000%
  test: n=1,362,520  normal=0  attack=1,362,520  attack_rate=100.0000%
  Обидва класи присутні у ВСІХ непорожніх сегментах: False

--- Часові частки (не квантилі рядків): train_end=2025-02-09 12:51:24, val_end=2025-02-11 02:46:44 ---
  train: n=35,006  normal=35,006  attack=0  attack_rate=0.0000%
  val: n=22,022  normal=0  attack=22,022  attack_rate=100.0000%
  test: n=1,415,105  normal=0  attack=

In [5]:
# ---------- ПІДСУМОК: що потрібно для остаточного рішення (не автоматично тут) ----------
print("=" * 70)
print("ПІДСУМОК Step 5C.3")
print("=" * 70)
print("Вище показано: (1) daily/hourly розподіл, (2) коли зникає Normal, (3) кандидатні")
print("часові межі за ЧАСОВОЮ ЧАСТКОЮ діапазону (не квантилем рядків), кілька варіантів.")
print("\nОстаточний train_end/val_end для GeNIS ще НЕ зафіксовано — обирається за цим")
print("виводом, з умовою: обидва класи присутні в train, val І test.")

ПІДСУМОК Step 5C.3
Вище показано: (1) daily/hourly розподіл, (2) коли зникає Normal, (3) кандидатні
часові межі за ЧАСОВОЮ ЧАСТКОЮ діапазону (не квантилем рядків), кілька варіантів.

Остаточний train_end/val_end для GeNIS ще НЕ зафіксовано — обирається за цим
виводом, з умовою: обидва класи присутні в train, val І test.
